In [1]:
# 1. Cài đặt thư viện cần thiết (nếu thiếu)

import subprocess
import sys

# Cài gymnasium (thường chưa có sẵn trên Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

print("Thư viện đã sẵn sàng.")

Thư viện đã sẵn sàng.


## 2. Clone GitHub Repo vào `/kaggle/working/repo`

Sử dụng Kaggle Secret `GH_TOKEN` để clone repo chứa mã nguồn PPO + LSTM.

In [2]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import sys

# =============================================================
# CẬP NHẬT TÊN REPO TẠI ĐÂY
# =============================================================
GITHUB_REPO  = "sin0235/Project" 

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GH_TOKEN")
BRANCH       = "master"


import subprocess, sys
from pathlib import Path

CLONE_DIR   = Path("/kaggle/working/repo")
WORKING_DIR = Path("/kaggle/working")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull thất bại. Kiểm tra lại TOKEN và REPO.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Cloning into '/kaggle/working/repo'...

Project root: /kaggle/working/repo


## 3. Đọc cấu hình PPO từ `Conf/ppo_conf.yaml`

File config nằm trong repo vừa clone, dùng để điều chỉnh nhanh hyperparameter khi train trên Kaggle.

In [ ]:
import yaml
import numpy as np
from pprint import pprint

CONF_PATH = PROJECT_ROOT / "Conf" / "ppo_conf.yaml"

with open(CONF_PATH, "r", encoding="utf-8") as f:
    ppo_cfg = yaml.safe_load(f)

# -------------------------------------------------------------
# OVERRIDE duong dan data khi chay tren Kaggle
# -------------------------------------------------------------
# Dataset Kaggle v2: /kaggle/input/datasets/phctontrn/dataset-trading-rl-v2
# Gia dinh cac file ACB.csv, BID.csv,... nam truc tiep trong thu muc nay.
ppo_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl-v2"

print("Duong dan config:", CONF_PATH)
print("Data path (Kaggle v2):", ppo_cfg["data_path"])
print("\nCau hinh PPO (rut gon):")
keys_to_show = [
    "tickers", "features", "window_size",
    "train_ratio", "val_ratio", "test_ratio",
    "initial_balance", "fee_rate", "reward_name", "reward_window",
    "reward_excess_scale", "reward_advanced_alpha", "reward_advanced_beta", "reward_advanced_gamma",
    "trade_deadband", "max_weight_change_per_step", "dirichlet_total_concentration",
    "learning_rate", "n_steps", "batch_size", "n_epochs", "clip_range", "ent_coef", "target_kl", "total_timesteps",
]
summary = {k: ppo_cfg.get(k) for k in keys_to_show}
pprint(summary)


## 4. Chạy training PPO + LSTM

Cell dưới sẽ gọi trực tiếp `train_ppo(config)` trong `src/training/PPO.py`.

- Logger sẽ tự tạo `run_id` và ghi log vào `results/runs/<run_id>` bên trong repo
- Khi muốn thử hyperparameters khác, chỉ cần chỉnh `Conf/ppo_conf.yaml` rồi chạy lại cell này.

In [ ]:
import os
import sys

# Dam bao PROJECT_ROOT da nam trong sys.path tu cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("CWD:", os.getcwd())

from src.training.PPO import train_ppo

# Co the override them mot so hyperparameters ngay tai notebook neu muon, vi du:
# ppo_cfg["total_timesteps"] = 300_000
# ppo_cfg["learning_rate"] = 1e-3
# ppo_cfg["target_kl"] = 0.10

agent, test_metrics = train_ppo(ppo_cfg)

print("\nKet qua test trung binh (tom tat):")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")


## 5. Eval chi tiết và xem lại kết quả

Cell dưới cho phép bạn **chạy lại eval** (không train thêm) từ checkpoint phù hợp nhất của run gần nhất: ưu tiên `best_model.pt`, nếu thiếu thì fallback sang `final_model.pt` hoặc `checkpoint_*.pt`, rồi in full metrics để tiện so sánh giữa các run.

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Dam bao PROJECT_ROOT da duoc set o cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import (
    build_reward_kwargs_from_config,
    get_results_root_candidates,
    resolve_eval_run_across_roots,
    resolve_ppo_config,
)
from src.utils.metrics import compute_all, format_report
from src.environment.trading_env import TradingEnv
from src.utils.data_splitter import load_data, split_by_ratio

# Eval lai: tu tim run moi nhat con dung duoc.
# Uu tien config.json cua run; neu thieu se co suy luan shape model tu checkpoint.

from src.models.lstm import PPOLSTMActorCritic
from src.agents.ppo_agent import PPOAgent

base_cfg = resolve_ppo_config()
base_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl-v2"

candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
resolved = resolve_eval_run_across_roots(
    candidate_roots,
    base_config=base_cfg,
    overrides={
        "data_path": "/kaggle/input/datasets/phctontrn/dataset-trading-rl-v2",
        "device": "auto",
    },
)
if resolved["run_dir"] is None:
    print("Khong tim thay run nao du artifact de eval.")
    if resolved["checked_results_roots"]:
        print("Da kiem tra cac thu muc:")
        for root in resolved["checked_results_roots"]:
            print(" -", root)
    if resolved["missing_results_roots"]:
        print("Thieu thu muc:")
        for root in resolved["missing_results_roots"]:
            print(" -", root)
    if resolved["all_skipped_runs"]:
        print("Cac run bi bo qua:")
        for item in resolved["all_skipped_runs"]:
            print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])
else:
        latest_run = resolved["run_dir"]
        ckpt_path = resolved["ckpt_path"]
        ckpt_source = resolved["ckpt_source"]
        cfg = resolved["config"]
        print("Results root:", resolved["results_root"])
        print("Latest evaluable run:", latest_run.name)
        print("Config source:", resolved["config_source"])
        print("Checkpoint source:", ckpt_source)
        if resolved["all_skipped_runs"]:
            print("Skipped runs during search:")
            for item in resolved["all_skipped_runs"]:
                print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])

        data_dict = load_data(tickers=cfg["tickers"], data_path=cfg["data_path"])
        split = split_by_ratio(
            data_dict,
            train_ratio=cfg["train_ratio"],
            val_ratio=cfg["val_ratio"],
            test_ratio=cfg["test_ratio"],
        )
        print("Data split summary:", split.summary())

        test_env = TradingEnv(
            tickers=cfg["tickers"],
            mode="continuous",
            initial_balance=cfg["initial_balance"],
            fee_rate=cfg["fee_rate"],
            window_size=cfg["window_size"],
            data_dict=split.test,
            features=cfg["features"],
            max_steps=cfg["max_steps_eval"],
            random_start=False,
            reward_scaling=cfg["reward_scaling"],
            reward_name=cfg["reward_name"],
            reward_kwargs=build_reward_kwargs_from_config(cfg),
            trade_deadband=cfg["trade_deadband"],
            max_weight_change_per_step=cfg["max_weight_change_per_step"],
            print_verbosity=999999,
        )

        state_space = test_env.state_space
        n_stocks = state_space.n_stocks
        n_features = state_space.n_features

        model = PPOLSTMActorCritic(
            n_stocks=n_stocks,
            n_features=n_features,
            seq_len=cfg["window_size"],
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
            log_std_init=cfg["log_std_init"],
            dirichlet_total_concentration=cfg["dirichlet_total_concentration"],
        )

        agent = PPOAgent(
            model=model,
            lr=cfg["learning_rate"],
            gamma=cfg["gamma"],
            gae_lambda=cfg["gae_lambda"],
            clip_range=cfg["clip_range"],
            ent_coef=cfg["ent_coef"],
            vf_coef=cfg["vf_coef"],
            max_grad_norm=cfg["max_grad_norm"],
            target_kl=cfg["target_kl"],
            n_epochs=cfg["n_epochs"],
            batch_size=cfg["batch_size"],
            device=cfg["device"],
        )

        print(f"Dang load checkpoint ({ckpt_source}):", ckpt_path)
        agent.load(str(ckpt_path))

        # Danh gia tren test set
        print("\nDang eval tren test set...")
        values_list = agent.evaluate(test_env, state_space, n_episodes=cfg["n_eval_episodes"])
        metrics_list = [compute_all(v, cfg["initial_balance"]) for v in values_list]

        avg_metrics = {}
        for key in metrics_list[0]:
            vals = [m[key] for m in metrics_list if key in m]
            avg_metrics[key] = float(np.mean(vals))

        print("\n=== TEST METRICS (TRUNG BINH) ===")
        print(format_report(avg_metrics))


In [6]:
import shutil
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import get_results_root_candidates

# Zip toàn bộ thư mục runs để tải/lưu trữ nhanh
candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
runs_dir = next((root for root in candidate_roots if root.exists() and any(root.iterdir())), None)

if runs_dir is not None:
    zip_base = runs_dir.parent / "runs_backup"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=runs_dir)
    print(f"Da tao file zip: {zip_path}")
    print(f"Source runs dir: {runs_dir}")
else:
    print("Khong tim thay thu muc runs nao co du lieu.")
    print("Da kiem tra:")
    for root in candidate_roots:
        print(" -", root)

Da tao file zip: /kaggle/working/repo/results/runs_backup.zip
Source runs dir: /kaggle/working/repo/results/runs
